In [210]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [211]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mohaiminulhoque/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mohaiminulhoque/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/mohaiminulhoque/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

##### Load Dataset

In [212]:
df = pd.read_excel("data.xlsx")
print(df.head())
print(df.columns.tolist())
print(df.shape)

   inline-comment-id  # Comment  \
0  84326dd1_566c7146          1   
1  84326dd1_566c7146          2   
2  99d1f8e4_92b31cea          3   
3  193d089f_f5fac752          4   
4  50c2f81e_ac4fd6fc          5   

                                            Question               Final Label  
0  is this what they intended? don't they really ...  request for confirmation  
1  is this what they intended? don't they really ...                  surprise  
2  Don't we need to increment 'i' in the else cas...                suggestion  
3  i can't see anywhere where this is set to fals...                suggestion  
4  are you sure you want to include this source f...                 criticism  
['inline-comment-id', '# Comment', 'Question', 'Final Label']
(499, 4)


##### Data Processing

In [213]:
# Cleaning
df.drop(columns=['inline-comment-id', '# Comment'], inplace=True)
df.rename(columns={'Question': 'question', 'Final Label': 'label'}, inplace=True)
df = df[['question', 'label']].dropna()
df = df[df['label'] != 'discarded']


In [214]:
# Mapping
label_mapping = {
    # Suggestions
    'suggestion': 'Suggestions',
    
    # Requests
    'action': 'Requests',
    'request for action': 'Requests',
    'request for confirmation': 'Requests',
    'request for information': 'Requests',
    'request for rationale': 'Requests',
    'request for clarification': 'Requests',
    'request for opinion': 'Requests',
    
    # Attitudes and emotions
    'criticism': 'Attitudes and emotions',
    'anger': 'Attitudes and emotions',
    'surprise': 'Attitudes and emotions',
    
    # Hypothetical scenario
    'hypothetical scenario': 'Hypothetical scenario',
    
    # Rhetorical questions
    'rhetorical question': 'Rhetorical questions'
}
df['label'] = df['label'].replace(label_mapping)
df = df.dropna(subset=['label'])

In [215]:
# Question Processing
df['question'] = df['question'].str.lower()

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def cleanup_question(text):
    # Check if the text is a valid string (handles missing values)
    if not isinstance(text, str):
        return ""

    # Lowercasing
    text = text.lower()
    
    # Removing Punctuation (Keeps only alphanumeric characters and spaces)
    text = re.sub(r'[^\w\s]', '', text)
    
    # Split the text into individual words
    words = text.split()
    
    # Remove Stopwords AND Lemmatize
    cleaned_words = []
    for word in words:
        if word not in stop_words:
            # Reduce word to its root form
            root_word = lemmatizer.lemmatize(word)
            cleaned_words.append(root_word)
            
    # Re-join the cleaned words back into a single string
    return " ".join(cleaned_words)

df['formatted_question'] = df['question'].apply(cleanup_question)

In [216]:
# Word Count
df['word_count'] = df['question'].apply(lambda text: len(str(text).split()))
df['character_count'] = df['question'].apply(lambda text: len(str(text)))

# Punctuation Count
df['question_mark_count'] = df['question'].apply(lambda text: str(text).count('?'))
df['punctuation_mark_count'] = df['question'].apply(lambda text: str(text).count('!'))

# if Word starts with wh
df['starts_with_wh'] = df['question'].apply(lambda text: 1 if text.startswith("Wh") else 0)

In [217]:
# Sentiment Analysis
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    text = str(text)
    scores = analyzer.polarity_scores(text)
    return scores['compound']

df['sentiment_score'] = df['question'].apply(get_sentiment)

In [218]:
### Vectorization
tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2
)

In [219]:
# Re-Ordering the column order
new_order = [
    'question', 
    'formatted_question', 
    'word_count', 
    'character_count', 
    'question_mark_count', 
    'punctuation_mark_count', 
    'starts_with_wh', 
    'sentiment_score',
    'label' 
]

df = df[new_order]

### Question - 1

In [220]:
X = tfidf.fit_transform(df['formatted_question'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [221]:
clf = RandomForestClassifier(
  n_estimators=200,
  class_weight='balanced',
  random_state=42,
  n_jobs=-1
)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [222]:
y_pred = clf.predict(X_test)
print("=" * 60)
print("TASK 1 RESULTS — Words + Preprocessing, No Extra Feature")
print("=" * 60)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
print(cm_df)

TASK 1 RESULTS — Words + Preprocessing, No Extra Feature

Accuracy: 0.5565

Classification Report:
                        precision    recall  f1-score   support

Attitudes and emotions       0.00      0.00      0.00        10
 Hypothetical scenario       0.00      0.00      0.00         3
              Requests       0.69      0.61      0.65        69
  Rhetorical questions       0.00      0.00      0.00         1
           Suggestions       0.50      0.66      0.57        41

              accuracy                           0.56       124
             macro avg       0.24      0.25      0.24       124
          weighted avg       0.55      0.56      0.55       124


Confusion Matrix:
                        Attitudes and emotions  Hypothetical scenario  \
Attitudes and emotions                       0                      0   
Hypothetical scenario                        0                      0   
Requests                                     2                      2   
Rhetorical 

### Question - 02

In [223]:
# Tokenizing using raw question
X_tfidf = tfidf.fit_transform(df["question"])

# Extract custom features
custom_features = df[[
    'word_count', 
    'character_count', 
    'question_mark_count', 
    'punctuation_mark_count', 
    'starts_with_wh', 
    'sentiment_score'
]].values

# Merge TF-IDF matrix with custom features
X = hstack([X_tfidf, custom_features])
y = df['label']

print(f"X shape: {X.shape}")
print(f"Features: {X_tfidf.shape[1]} TF-IDF + 6 custom = {X.shape[1]} total")

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

X shape: (494, 3006)
Features: 3000 TF-IDF + 6 custom = 3006 total


In [224]:
clf = RandomForestClassifier(
  n_estimators=200,
  class_weight='balanced',
  random_state=42,
  n_jobs=-1
)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [225]:
# ============================================================
# Evaluation
# ============================================================

y_pred = clf.predict(X_test)

print("=" * 65)
print("TASK 2 RESULTS — Words + Extra Features, NO Preprocessing")
print("=" * 65)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
print(cm_df)

TASK 2 RESULTS — Words + Extra Features, NO Preprocessing

Accuracy: 0.6048

Classification Report:
                        precision    recall  f1-score   support

Attitudes and emotions       0.00      0.00      0.00        10
 Hypothetical scenario       0.00      0.00      0.00         3
              Requests       0.64      0.81      0.71        69
  Rhetorical questions       0.00      0.00      0.00         1
           Suggestions       0.66      0.46      0.54        41

              accuracy                           0.60       124
             macro avg       0.26      0.26      0.25       124
          weighted avg       0.57      0.60      0.58       124


Confusion Matrix:
                        Attitudes and emotions  Hypothetical scenario  \
Attitudes and emotions                       0                      0   
Hypothetical scenario                        0                      0   
Requests                                     0                      2   
Rhetorical

In [226]:
# ============================================================
# Evaluation
# ============================================================

print("=" * 65)
print("TASK 2 RESULTS — Words + Extra Features, NO Preprocessing")
print("=" * 65)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
print(cm_df)

TASK 2 RESULTS — Words + Extra Features, NO Preprocessing

Accuracy: 0.6048

Classification Report:
                        precision    recall  f1-score   support

Attitudes and emotions       0.00      0.00      0.00        10
 Hypothetical scenario       0.00      0.00      0.00         3
              Requests       0.64      0.81      0.71        69
  Rhetorical questions       0.00      0.00      0.00         1
           Suggestions       0.66      0.46      0.54        41

              accuracy                           0.60       124
             macro avg       0.26      0.26      0.25       124
          weighted avg       0.57      0.60      0.58       124


Confusion Matrix:
                        Attitudes and emotions  Hypothetical scenario  \
Attitudes and emotions                       0                      0   
Hypothetical scenario                        0                      0   
Requests                                     0                      2   
Rhetorical

### Question - 03

In [227]:
# Tokenizing using raw question
X_tfidf = tfidf.fit_transform(df["formatted_question"])

# Extract custom features
custom_features = df[[
    'word_count', 
    'character_count', 
    'question_mark_count', 
    'punctuation_mark_count', 
    'starts_with_wh', 
    'sentiment_score'
]].values

# Merge TF-IDF matrix with custom features
X = hstack([X_tfidf, custom_features])
y = df['label']

print(f"X shape: {X.shape}")
print(f"Features: {X_tfidf.shape[1]} TF-IDF + 6 custom = {X.shape[1]} total")

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

X shape: (494, 1944)
Features: 1938 TF-IDF + 6 custom = 1944 total


In [228]:
clf = RandomForestClassifier(
  n_estimators=200,
  class_weight='balanced',
  random_state=42,
  n_jobs=-1
)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [229]:
# ============================================================
# Evaluation
# ============================================================

y_pred = clf.predict(X_test)

print("=" * 65)
print("TASK 3 RESULTS — Words + Extra Features, WITH Preprocessing")
print("=" * 65)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
print(cm_df)

TASK 3 RESULTS — Words + Extra Features, WITH Preprocessing

Accuracy: 0.5484

Classification Report:
                        precision    recall  f1-score   support

Attitudes and emotions       0.00      0.00      0.00        10
 Hypothetical scenario       0.00      0.00      0.00         3
              Requests       0.61      0.80      0.69        69
  Rhetorical questions       0.00      0.00      0.00         1
           Suggestions       0.52      0.32      0.39        41

              accuracy                           0.55       124
             macro avg       0.23      0.22      0.22       124
          weighted avg       0.51      0.55      0.52       124


Confusion Matrix:
                        Attitudes and emotions  Hypothetical scenario  \
Attitudes and emotions                       0                      0   
Hypothetical scenario                        0                      0   
Requests                                     1                      2   
Rhetoric

### Question - 04

In [230]:
from imblearn.over_sampling import SMOTE

# Split the data first
X = df.iloc[:, :-1]  # All columns except label
y = df['label']
X_train_df, X_test_df, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [234]:
# Create a fresh TF-IDF vectorizer for Question 4
tfidf_q4 = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2
)

# Tokenizing using formatted question
X_train_tfidf = tfidf_q4.fit_transform(X_train_df["formatted_question"])

# Extract custom features
custom_features_train = X_train_df[[
    'word_count',
    'character_count',
    'question_mark_count',
    'punctuation_mark_count',
    'starts_with_wh',
    'sentiment_score'
]].values

# Merge TF-IDF matrix with custom features
X_train_combined = hstack([X_train_tfidf, custom_features_train])

# Apply SMOTE resampling (reduce k_neighbors for small classes)
smote = SMOTE(random_state=42, k_neighbors=2)  # Use fewer neighbors
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_combined, y_train)

print(f"Original training shape: {X_train_combined.shape}")
print(f"Resampled training shape: {X_train_resampled.shape}")
print(f"Original class distribution: {y_train.value_counts().to_dict()}")
print(f"Resampled class distribution: {pd.Series(y_train_resampled).value_counts().to_dict()}")

Original training shape: (370, 1380)
Resampled training shape: (1025, 1380)
Original class distribution: {'Requests': 205, 'Suggestions': 122, 'Attitudes and emotions': 28, 'Hypothetical scenario': 10, 'Rhetorical questions': 5}
Resampled class distribution: {'Suggestions': 205, 'Requests': 205, 'Hypothetical scenario': 205, 'Rhetorical questions': 205, 'Attitudes and emotions': 205}


In [232]:
clf = RandomForestClassifier(
  n_estimators=200,
  class_weight='balanced',
  random_state=42,
  n_jobs=-1
)
clf.fit(X_train_resampled, y_train_resampled)

RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)

In [235]:
# ============================================================
# Evaluation
# ============================================================

# Tokenizing using formatted question (use transform with the same vectorizer)
X_test_tfidf = tfidf_q4.transform(X_test_df["formatted_question"])

# Extract custom features
custom_features_test = X_test_df[[
    'word_count',
    'character_count',
    'question_mark_count',
    'punctuation_mark_count',
    'starts_with_wh',
    'sentiment_score'
]].values

# Merge TF-IDF matrix with custom features
X_test_combined = hstack([X_test_tfidf, custom_features_test])

y_pred = clf.predict(X_test_combined)

print("=" * 65)
print("TASK 4 RESULTS — Words + Extra Features, WITH Preprocessing, WITH Resampling")
print("=" * 65)
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
cm_df = pd.DataFrame(cm, index=clf.classes_, columns=clf.classes_)
print(cm_df)

TASK 4 RESULTS — Words + Extra Features, WITH Preprocessing, WITH Resampling

Accuracy: 0.5887

Classification Report:
                        precision    recall  f1-score   support

Attitudes and emotions       0.00      0.00      0.00        10
 Hypothetical scenario       0.00      0.00      0.00         3
              Requests       0.63      0.86      0.73        69
  Rhetorical questions       0.00      0.00      0.00         1
           Suggestions       0.67      0.34      0.45        41

              accuracy                           0.59       124
             macro avg       0.26      0.24      0.24       124
          weighted avg       0.57      0.59      0.55       124


Confusion Matrix:
                        Attitudes and emotions  Hypothetical scenario  \
Attitudes and emotions                       0                      0   
Hypothetical scenario                        0                      0   
Requests                                     2                  